# 05 — Export for Looker Studio

In [1]:
print('TODO')

TODO


In [2]:
from pathlib import Path
import pandas as pd, subprocess

pd.options.display.float_format = '{:,.2f}'.format

# Find the real repo root (works no matter where the notebook lives)
try:
    repo_root = subprocess.check_output(
        ["git", "rev-parse", "--show-toplevel"]
    ).decode().strip()
except Exception:
    # fallback: assume this notebook lives in <repo>/notebooks
    repo_root = str(Path.cwd().parent)

REPO = Path(repo_root)
DS = REPO / "analytics" / "looker_studio_datasources"

EXPECTED = [
    "counts_by_hour.csv",
    "daily_totals.csv",
    "heatmap_weekday_hour.csv",
    "power_hours_top3.csv",
    "rain_uplift_by_sensor.csv",
    "sensors_lookup.csv",
]

print("Repo:", REPO)
print("Data sources dir:", DS)
print("Exists?", DS.exists())
print("Files:", sorted(p.name for p in DS.glob("*.csv")))


Repo: /Users/poojithraj/Documents/melbourne-foot-traffic-marketing
Data sources dir: /Users/poojithraj/Documents/melbourne-foot-traffic-marketing/analytics/looker_studio_datasources
Exists? True
Files: ['counts_by_hour.csv', 'daily_totals.csv', 'heatmap.csv', 'heatmap_weekday_hour.csv', 'monthly_totals.csv', 'power_hours.csv', 'power_hours_top3.csv', 'rain_uplift_by_sensor.csv', 'sensors_lookup.csv']


In [3]:
# --- Load & normalize headers ---
from pathlib import Path
import pandas as pd

def load_csv_clean(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, encoding="utf-8-sig")
    # drop stray index columns
    drop = [c for c in df.columns if c.lower().startswith("unnamed")]
    if drop:
        df = df.drop(columns=drop)
    # ASCII snake_case headers
    df.columns = (
        df.columns
          .str.strip()
          .str.replace("[^0-9a-zA-Z]+", "_", regex=True)
          .str.strip("_")
          .str.lower()
    )
    return df

missing = [f for f in EXPECTED if not (DS/f).exists()]
assert not missing, f"Missing CSVs: {missing}"

tables = {name.replace(".csv",""): load_csv_clean(DS/name) for name in EXPECTED}
{k: v.shape for k, v in tables.items()}


{'counts_by_hour': (3554, 5),
 'daily_totals': (153, 5),
 'heatmap_weekday_hour': (15626, 7),
 'power_hours_top3': (288, 8),
 'rain_uplift_by_sensor': (96, 4),
 'sensors_lookup': (135, 2)}

In [4]:
from pandas.api.types import is_numeric_dtype
import pandas as pd

def coerce_types(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    # common date/time fields
    for cand in ["date", "sensing_date", "day", "dt", "datetime"]:
        if cand in out.columns:
            out[cand] = pd.to_datetime(out[cand], errors="coerce").dt.tz_localize(None)
    # discrete ints
    for cand in ["hour", "weekday", "hourday", "dow"]:
        if cand in out.columns:
            out[cand] = pd.to_numeric(out[cand], errors="coerce").astype("Int64")
    # metric columns to numeric where sensible
    for col in out.columns:
        if not is_numeric_dtype(out[col]) and any(k in col for k in [
            "count","median","mean","total","uplift","rain","wind","temp","precip"
        ]):
            out[col] = pd.to_numeric(out[col], errors="ignore")
    return out

tables = {k: coerce_types(v) for k, v in tables.items()}
{k: v.dtypes.head(10) for k, v in tables.items()}


{'counts_by_hour': date       datetime64[ns]
 hour                Int64
 count               int64
 weekday             Int64
 month              object
 dtype: object,
 'daily_totals': date            datetime64[ns]
 total_counts             int64
 hour                     Int64
 weekday                  Int64
 month                   object
 dtype: object,
 'heatmap_weekday_hour': sensor_id         int64
 dow_name         object
 hour              Int64
 median_count    float64
 mean_count      float64
 samples           int64
 weekday           Int64
 dtype: object,
 'power_hours_top3': sensor_id         int64
 dow_name         object
 hour              Int64
 median_count    float64
 mean_count      float64
 samples           int64
 rank              int64
 weekday           Int64
 dtype: object,
 'rain_uplift_by_sensor': sensor_id        int64
 med_no_rain    float64
 med_rain       float64
 uplift_pct     float64
 dtype: object,
 'sensors_lookup': sensor_id       int64
 sensor_na

In [5]:
import pandas as pd

def weekday_from_name(series: pd.Series) -> pd.Series:
    # robust map: Mon/Tue/Wed... or Monday/Tuesday...
    map3 = {"mon":0,"tue":1,"wed":2,"thu":3,"fri":4,"sat":5,"sun":6}
    return (series.astype(str)
                  .str.strip()
                  .str[:3]
                  .str.lower()
                  .map(map3)
                  .astype("Int64"))

def add_project_helpers(name: str, df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()

    # --- time primitives ---
    # prefer 'date' if present, else derive from 'date_time'
    if "date" in out.columns:
        dt = pd.to_datetime(out["date"], errors="coerce")
    elif "date_time" in out.columns:
        dt = pd.to_datetime(out["date_time"], errors="coerce")
        out["date"] = dt.dt.date.astype("string")  # Looker Date ingest is fine from yyyy-mm-dd
    else:
        dt = None

    if dt is not None:
        if "hour" not in out.columns:
            out["hour"] = dt.dt.hour.astype("Int64")
        if "weekday" not in out.columns:
            out["weekday"] = dt.dt.dayofweek.astype("Int64")
        if "month" not in out.columns:
            out["month"] = dt.dt.to_period("M").astype(str)

    # fill weekday if it exists as text (dow_name) but not numeric
    if "weekday" not in out.columns and "dow_name" in out.columns:
        out["weekday"] = weekday_from_name(out["dow_name"])

    # --- rain flag ---
    if "is_rain" not in out.columns:
        if "rain_flag" in out.columns:
            out["is_rain"] = pd.to_numeric(out["rain_flag"], errors="coerce").fillna(0).astype(int)
        elif "rain" in out.columns:
            out["is_rain"] = (pd.to_numeric(out["rain"], errors="coerce").fillna(0) > 0).astype(int)

    # --- friendly metric aliases for Looker charts ---
    # counts_by_hour: make a generic 'count'
    if name == "counts_by_hour" and "hourly_counts" in out.columns and "count" not in out.columns:
        out["count"] = out["hourly_counts"]

    # daily_totals: expose 'total_counts'
    if name == "daily_totals" and "daily_total" in out.columns and "total_counts" not in out.columns:
        out["total_counts"] = out["daily_total"]

    # ensure sensor_id exists everywhere it should
    if "sensor_id" in out.columns:
        out["sensor_id"] = pd.to_numeric(out["sensor_id"], errors="coerce").astype("Int64")

    return out

tables = {k: add_project_helpers(k, v) for k, v in tables.items()}
{ k: v.head(3) for k, v in tables.items() }


{'counts_by_hour':         date  hour  count  weekday    month
 0 2025-03-01     0  12622        5  2025-03
 1 2025-03-01     1   8264        5  2025-03
 2 2025-03-01     2   5400        5  2025-03,
 'daily_totals':         date  total_counts  hour  weekday    month
 0 2025-03-01        929244     0        5  2025-03
 1 2025-03-02        670821     0        6  2025-03
 2 2025-03-03        798301     0        0  2025-03,
 'heatmap_weekday_hour':    sensor_id dow_name  hour  median_count  mean_count  samples  weekday
 0          1   Monday     0         51.00       61.00        5        0
 1          2   Monday     0         48.00       54.60        5        0
 2          3   Monday     0        185.00      226.60        5        0,
 'power_hours_top3':    sensor_id dow_name  hour  median_count  mean_count  samples  rank  weekday
 0        180   Monday     8        144.00      138.33        3     3        0
 1         68   Monday    12      1,126.00      989.40        5     2        0
 2

In [6]:
# Sort consistently so diffs are stable
for name, df in tables.items():
    out = df.copy()
    sort_keys = [c for c in ["date","weekday","hour","sensor_id","sensor_name"] if c in out.columns]
    if sort_keys:
        out = out.sort_values(sort_keys)
    out.to_csv(DS / f"{name}.csv", index=False, encoding="utf-8", na_rep="")
    print("wrote:", (DS / f"{name}.csv").relative_to(REPO))


wrote: analytics/looker_studio_datasources/counts_by_hour.csv
wrote: analytics/looker_studio_datasources/daily_totals.csv
wrote: analytics/looker_studio_datasources/heatmap_weekday_hour.csv
wrote: analytics/looker_studio_datasources/power_hours_top3.csv
wrote: analytics/looker_studio_datasources/rain_uplift_by_sensor.csv
wrote: analytics/looker_studio_datasources/sensors_lookup.csv


In [7]:
# --- Sanity checks for exported tables ---

# 1. Column checks: make sure each table has what the dashboard expects
expected_tables = {
    # Aggregated across all sensors, so no sensor_id here
    "counts_by_hour": ["date", "weekday", "hour", "count"],
    "daily_totals": ["date", "total_counts"],
    "heatmap_weekday_hour": ["weekday", "hour", "median_count"],
    "power_hours_top3": ["weekday", "hour", "median_count", "rank"],
    "rain_uplift_by_sensor": ["sensor_id", "uplift_pct"],
    "sensors_lookup": ["sensor_id", "sensor_name"],
}

for name, cols in expected_tables.items():
    assert name in tables, f"Missing table in `tables`: {name!r}"
    missing = [c for c in cols if c not in tables[name].columns]
    assert not missing, f"{name} missing columns: {missing}"

# 2. Value / range checks for key fields
# counts_by_hour: hour 0–23, weekday 0–6 (if present)
tbl = tables["counts_by_hour"]
assert tbl["hour"].between(0, 23).all(), "counts_by_hour.hour must be 0–23"
if "weekday" in tbl.columns:
    assert tbl["weekday"].between(0, 6).all(), "counts_by_hour.weekday must be 0–6"

# heatmap_weekday_hour: hour 0–23, weekday 0–6
tbl = tables["heatmap_weekday_hour"]
assert tbl["hour"].between(0, 23).all(), "heatmap_weekday_hour.hour must be 0–23"
assert tbl["weekday"].between(0, 6).all(), "heatmap_weekday_hour.weekday must be 0–6"

# power_hours_top3: hour 0–23, rank 1–3 if present
tbl = tables["power_hours_top3"]
assert tbl["hour"].between(0, 23).all(), "power_hours_top3.hour must be 0–23"
if "rank" in tbl.columns:
    assert tbl["rank"].between(1, 3).all(), "power_hours_top3.rank must be 1–3"

print("✅ Sanity checks passed")


✅ Sanity checks passed


In [8]:
for name, df in tables.items():
    out = df.copy()
    sort_keys = [c for c in ["date","weekday","hour","sensor_id","sensor_name"] if c in out.columns]
    if sort_keys:
        out = out.sort_values(sort_keys)
    out.to_csv(DS / f"{name}.csv", index=False, encoding="utf-8", na_rep="")
    print("wrote:", (DS / f"{name}.csv").relative_to(REPO))


wrote: analytics/looker_studio_datasources/counts_by_hour.csv
wrote: analytics/looker_studio_datasources/daily_totals.csv
wrote: analytics/looker_studio_datasources/heatmap_weekday_hour.csv
wrote: analytics/looker_studio_datasources/power_hours_top3.csv
wrote: analytics/looker_studio_datasources/rain_uplift_by_sensor.csv
wrote: analytics/looker_studio_datasources/sensors_lookup.csv


In [9]:
# Old sanity-check logic removed.
# We now use the checks in the cell above (✅ Sanity checks passed).
# This cell is intentionally left as a no-op so the notebook runs clean.
pass


In [10]:
# Git automation disabled for this run.
# Data exports have already been written to analytics/looker_studio_datasources/.
# If you want to push to GitHub, do it manually from the terminal.
print("✅ Data exports updated. Skipping git commit/push from inside the notebook.")


✅ Data exports updated. Skipping git commit/push from inside the notebook.
